# 일강수로 연강수 지도를 어떻게 만드는가?

시간 합계와 공간 분포의 차이 이해

**할 일** · 2020년 일자료 수 확인 → 연합계 → 지역 분포 비교

위에서 아래로 실행. 코드를 바꾸기 전 결과를 예상하고, 실행 결과와 비교

## 실행 준비

CPU 런타임 사용. Earth Engine에 등록한 Cloud 프로젝트와 사용 권한 필요. [설정 안내](https://github.com/steelpl/lecture/blob/main/CE776-climate-change-carbon-water-cycle/GEE_SETUP.md) 참고. 데이터 조회에는 본인 계정 인증 필요

In [ ]:
%pip -q install earthengine-api==1.7.52 geemap==0.39.0

In [ ]:
import ee
import geemap

PROJECT_ID = ""  # Earth Engine에 등록한 본인의 Cloud 프로젝트 ID
if not PROJECT_ID.strip():
    raise ValueError("PROJECT_ID에 본인의 등록된 Cloud 프로젝트 ID 입력")
ee.Authenticate()
ee.Initialize(project=PROJECT_ID)

In [ ]:
roi = ee.Geometry.Rectangle([126.80, 37.43, 127.15, 37.70], geodesic=False)

In [ ]:
rain = (ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')
        .filterBounds(roi).filterDate('2020-01-01','2021-01-01').select('precipitation'))
annual = rain.sum().rename('annual_mm').clip(roi)
summary = annual.reduceRegion(ee.Reducer.mean().combine(ee.Reducer.minMax(), sharedInputs=True), roi, 5566, maxPixels=100_000)
hist = annual.reduceRegion(ee.Reducer.histogram(maxBuckets=20),roi,5566,maxPixels=100_000)

2020년은 윤년: 366개 일자료 예상. 일강수율 mm/day를 1일 간격으로 합산한 연강수량은 mm. 영상 수와 각 픽셀의 유효 관측 수를 함께 확인

In [ ]:
print('daily images (expected 366):', rain.size().getInfo())
print('valid day counts:', rain.count().reduceRegion(ee.Reducer.minMax(),roi,5566,maxPixels=100_000).getInfo())
print('annual summary (mm):', summary.getInfo())
print('spatial histogram:', hist.getInfo())

In [ ]:
Map = geemap.Map()
Map.centerObject(roi, 10)
Map.addLayer(annual, {'min':500,'max':2000,'palette':['white','cyan','blue']}, '2020 precipitation (mm)')
Map

이 히스토그램이 ‘매일의 강수량 분포’인지 ‘연강수량의 공간 분포’인지 설명하시오. 서울 주변의 작은 영역에서 0.05° 자료로 세밀한 동네 차이를 해석할 수 있는지 설명하시오. 영역을 넓혀 분포를 비교하시오.

## 자료와 해석 범위

- [CHIRPS v2 Daily](https://developers.google.com/earth-engine/datasets/catalog/UCSB-CHG_CHIRPS_DAILY)

관측 연도는 고정 사례의 기간. 수업 연도와 무관. 코드는 자료 처리 방법을 보여 주며 지도·수치는 인증 후 조회 결과로 판단